# [DistilBERT, a distilled verison of BERT: smaller, faster, cheaper and lighter](https://arxiv.org/pdf/1910.01108)

## Introduction

**Gap:** Pre-trained language models like BERT achieve strong performance but are large, slow, and memory-intensive, making them difficult to deploy in real-world or resource-constrained settings.

Prior knowledge distillation approaches mainly focus on task-specific compression **after fine-tuning**, which limits reusability and does not produce a general-purpose efficient pretrained model.

**Improvement**: DistilBERT proposes a method to pretrain a smaller transformer model using knowledge distillation from BERT, producing a foundational model that retains most of BERT’s performance while being significantly more efficient for both inference and downstream fine-tuning.

## Approach

DistilBERT trains a smaller “student” transformer to **mimic** a frozen pretrained BERT “teacher” using **multiple complementary losses**:

* **Language Modeling Loss (MLM):**
The student is trained on masked language modeling, similar to BERT, ensuring it still learns meaningful token-level representations from raw text
* **Distillation Loss (soft-target matching):**
The student matches the teacher’s output distribution (softmax probabilities over vocabulary)
* **Cosine Embedding / Hidden-State Alignment Loss**:
The student is encouraged to align its internal hidden representations with those of the teacher by maximizing cosine similarity between corresponding token-level hidden states.
This acts as an intermediate-level constraint, pushing the student not just to match outputs but also to learn similar feature spaces.


## Result

DistilBERT retains approximately 97% of BERT’s NLU performance while being ~40% smaller and ~60% faster at inference time.

## Application

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/toxicity_en.csv")
df

,text,is_toxic
0,"Elon Musk is a piece of shit, greedy capitalis...",Toxic
1,The senile credit card shrill from Delaware ne...,Toxic
2,He does that a lot -- makes everyone look good...,Toxic
3,F*ck Lizzo,Toxic
4,Epstein and trump were best buds!!! Pedophiles...,Toxic
...,...,...
995,My maternal abuelita taught me how to make pla...,Not Toxic
996,Funnily enough I was looking online last week ...,Not Toxic
997,I can't bear how nice this is.\n \n I guess it...,Not Toxic
998,Going to buy a share of Tesla just to ensure i...,Not Toxic


In [3]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load tokenizer and model (recommended generic classes)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

text = "Replace me by any text you'd like."

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Forward pass (no grad for inference)
with torch.no_grad():
    outputs = model(**inputs)

# Outputs
last_hidden_state = outputs.last_hidden_state
pooler_output = outputs.pooler_output  # if available

print(last_hidden_state.shape)
print(pooler_output.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 12, 768])
torch.Size([1, 768])


In [4]:
import torch.nn as nn 

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size=768, output_size=1):
        super().__init__()
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        return self.linear(x)

In [5]:
classifier = ClassifierHead()
classifier(last_hidden_state.sum(axis=1))

tensor([[-3.0503]], grad_fn=<AddmmBackward0>)

In [6]:
df.is_toxic.value_counts()

is_toxic
Toxic        501
Not Toxic    499
Name: count, dtype: int64

In [7]:
df['is_toxic'] = (df['is_toxic'] == 'Toxic').astype(int)
df.is_toxic

0      1
1      1
2      1
3      1
4      1
      ..
995    0
996    0
997    0
998    0
999    0
Name: is_toxic, Length: 1000, dtype: int32

In [8]:
d = tokenizer(df.text.to_list(), padding=True, truncation=True, return_tensors="pt")
d

{'input_ids': tensor([[  101,  3449,  2239,  ...,     0,     0,     0],
        [  101,  1996, 12411,  ...,     0,     0,     0],
        [  101,  2002,  2515,  ...,     0,     0,     0],
        ...,
        [  101,  1045,  2064,  ...,     0,     0,     0],
        [  101,  2183,  2000,  ...,     0,     0,     0],
        [  101,  1045,  2069,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [9]:
input_ids, attention_mask = d['input_ids'], d['attention_mask']

In [10]:
out = model(input_ids[:3,:], attention_mask=attention_mask[:3,:])
out

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[ 0.0593,  0.1049, -0.4640,  ..., -0.1336,  0.4188,  0.5371],
         [ 0.3524,  0.0425,  0.9339,  ...,  0.5419,  0.5637, -0.7458],
         [ 0.0203, -0.5564, -0.0551,  ...,  0.5582, -0.0117, -0.3692],
         ...,
         [ 0.2029, -0.0115,  0.1056,  ...,  0.0233,  0.0651, -0.2512],
         [ 0.0205,  0.1021,  0.0692,  ..., -0.0670, -0.0094,  0.0809],
         [ 0.1132, -0.1850,  0.4670,  ...,  0.0568, -0.0712, -0.2115]],

        [[ 0.0851,  0.3976, -0.0660,  ..., -0.2570,  0.1017,  0.4449],
         [-0.2085, -0.1601, -0.0868,  ..., -0.1615,  0.6663, -0.2199],
         [-0.6927,  0.2375,  0.9202,  ..., -0.7924,  0.2421, -0.4554],
         ...,
         [ 0.3492, -0.1504,  0.1246,  ...,  0.1441,  0.1747,  0.1430],
         [ 0.2518, -0.0267,  0.6830,  ...,  0.0362, -0.0680, -0.0566],
         [ 0.2300, -0.0646,  0.6237,  ...,  0.0830, -0.0835,  0.0137]],

        [[ 0.2009,  0.3101,  0.2339,  ..., -0.2317,  

In [40]:
from torch.nn import CrossEntropyLoss, BCELoss
from torch.optim import Adam
from torch import nn

criterion = BCELoss()
optimizer = Adam(classifier.parameters(), lr=5e-4)

In [48]:
out = model(input_ids[:3,:], attention_mask=attention_mask[:3,:])

logits = classifier(out.last_hidden_state.sum(axis=1)).squeeze(-1)
sigmoid = nn.functional.sigmoid(logits) 

labels = torch.tensor(df.is_toxic)[:3].to(torch.float)
labels = labels.to(torch.float)
loss = criterion(sigmoid, labels)
loss.backward()
optimizer.step()
print(loss)